# cudf-bench: groupby skew sweep (Step 3)

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Total ~10 min. **Click Allow when the browser asks to download at the end.**

Finding under investigation: cuDF `groupby_agg` gets **slower** on skewed keys while CPU libraries get faster. Two experiments:

- **A. Skew dial** — same data size, turn skew 0 → 2.0. If time rises smoothly with skew, contention hypothesis gains evidence; a cliff points at an algorithm switch.
- **B. Group-count control** — uniform keys but vary how many distinct groups (10 → 1M). Separates "few big groups" from "one hot group": if cuDF is also slow with few *uniform* groups, the problem is rows-per-group, not skew itself.

In [ ]:
!nvidia-smi -L

In [ ]:
try:
    import cudf
except ImportError:
    %pip install -q cudf-cu12
    import cudf
print("cudf", cudf.__version__)

In [ ]:
%cd /content
![ -d cudf-bench ] || git clone -q https://github.com/alexislowys/cudf-bench.git
%cd /content/cudf-bench
!git pull -q
!rm -f results/sweep.csv
%pip install -q -e .

## Experiment A: skew dial (10M rows, default 100k keys)

In [ ]:
!python -m benchmark.runner --backend cudf   --ops groupby_agg --rows 1e7 --skew 0,0.25,0.5,0.75,1.0,1.25,1.5,1.75,2.0 --out results/sweep.csv
!python -m benchmark.runner --backend polars --ops groupby_agg --rows 1e7 --skew 0,0.25,0.5,0.75,1.0,1.25,1.5,1.75,2.0 --out results/sweep.csv

## Experiment B: group-count control (uniform keys, 10M rows)

In [ ]:
for k in [10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    !python -m benchmark.runner --backend cudf   --ops groupby_agg --rows 1e7 --skew 0 --n-keys {k} --out results/sweep.csv
    !python -m benchmark.runner --backend polars --ops groupby_agg --rows 1e7 --skew 0 --n-keys {k} --out results/sweep.csv

## Quick look

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("results/sweep.csv")
sweep_a = df[df["n_keys"].isna()]
sweep_b = df[df["n_keys"].notna()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for backend, g in sweep_a.groupby("backend"):
    ax1.plot(g["skew"], g["median_s"], marker="o", label=backend)
ax1.set_xlabel("skew"); ax1.set_ylabel("median s"); ax1.set_title("A: groupby vs skew (10M rows)"); ax1.legend()
for backend, g in sweep_b.groupby("backend"):
    g = g.sort_values("n_keys")
    ax2.plot(g["n_keys"], g["median_s"], marker="o", label=backend)
ax2.set_xscale("log"); ax2.set_xlabel("distinct keys (uniform)"); ax2.set_ylabel("median s")
ax2.set_title("B: groupby vs group count (10M rows)"); ax2.legend()
plt.tight_layout(); plt.show()

## Auto-download (click Allow)

In [ ]:
from google.colab import files
files.download("/content/cudf-bench/results/sweep.csv")